# Breast Cancer SVM Hyperparameter Tuning

This notebook tunes the isolated SVM model on `data/raw/breast_cancer_dataset.csv` with cross-validation and saves results under `svm/results/`.

In [ ]:
import sys
from pathlib import Path

print("Python executable:", sys.executable)

current_path = Path.cwd().resolve()
repo_root = None
for path in [current_path, *current_path.parents]:
    if (path / "data" / "raw" / "breast_cancer_dataset.csv").exists() and (path / "svm" / "tune_svm.py").exists():
        repo_root = path
        break

if repo_root is None:
    raise FileNotFoundError("Could not find the project root from this notebook location.")

sys.path.insert(0, str(repo_root / "svm"))

try:
    import pandas as pd
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "pandas is not installed in this notebook kernel. Select the project "
        "virtualenv kernel/interpreter: .venv/bin/python"
    ) from exc

try:
    from tune_svm import DEFAULT_DATA_PATH, DEFAULT_RESULTS_DIR, tune_svm
except ImportError as exc:
    raise RuntimeError(
        "Could not import the SVM tuning dependencies. In Jupyter, select the project "
        "virtualenv kernel/interpreter: .venv/bin/python"
    ) from exc

print("Project root:", repo_root)
print("Data path:", DEFAULT_DATA_PATH)

In [ ]:
df = pd.read_csv(DEFAULT_DATA_PATH)

print("Shape:", df.shape)
display(df.head())
display(df["diagnosis"].value_counts().rename_axis("diagnosis").to_frame("count"))

In [ ]:
metrics = tune_svm(
    data_path=DEFAULT_DATA_PATH,
    results_dir=DEFAULT_RESULTS_DIR,
    test_size=0.2,
    random_state=42,
    cv_folds=3,
    scoring="roc_auc",
)

run_dir = DEFAULT_RESULTS_DIR / metrics["run_id"]
run_dir

In [ ]:
pd.Series(
    {
        "target_column": metrics["target_column"],
        "positive_class": metrics["positive_class"],
        "best_params": metrics["best_params"],
        "best_cv_score": metrics["best_cv_score"],
        "accuracy": metrics["accuracy"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "roc_auc": metrics["roc_auc"],
    }
).to_frame("value")

In [ ]:
cv_results = pd.read_csv(run_dir / "cv_results.csv")
cv_results[
    [
        "rank_test_score",
        "mean_test_score",
        "std_test_score",
        "param_model__C",
        "param_model__gamma",
        "param_model__kernel",
    ]
].head(10)